# 02. Network Layouts and Node Overlap

This notebook studies a practical layout-design problem:

**How do we move from a default graph layout to a readable layout when nodes overlap?**

It follows `01-networkx-drawing-basics.ipynb` and focuses on what happens once the graph is dense enough that node position becomes a real design problem.

The notebook is organized as a workflow rather than a catalog of parameters:
1. warm up with the main NetworkX layouts on a tiny graph
2. inspect why overlap appears once node size encodes importance
3. tune a standard NetworkX layout before inventing new machinery
4. add a pure-Python **size-aware local overlap-removal pass**
5. compare that result with a Graphviz alternative

Three ideas stay separate throughout:
- **layout quality**: does the geometry reflect something meaningful about the graph?
- **overlap reduction**: do nodes occupy distinct space instead of sitting on top of each other?
- **overall readability**: does the final figure become easier to interpret, or just more spread out?

The goal is not to chase a perfect layout. The goal is to learn a disciplined way to improve readability while keeping the structure honest.

Continue with `03-network-labels.ipynb` for the next dense-graph problem: label overlap.


In [ ]:
from pathlib import Path
import math
import sys
import warnings

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
tutorials_dir = cwd.parent if cwd.name == "04-networks" else cwd / "tutorials"
if str(tutorials_dir) not in sys.path:
    sys.path.insert(0, str(tutorials_dir))

from dataviz_utils import *

set_seeds()
TEXT = apply_teaching_rc(grid=False, font_base=11.5)

LAYOUT_STYLE = {
    "node_fill": "white",
    "node_edge": "#1F2933",
    "node_linewidth": 0.85,
    "edge_color": "#C9D2DC",
    "edge_alpha": 0.6,
    "edge_width": 0.85,
    "highlight_fill": lighten_color(DV_PALETTE["gold"], 0.80),
    "highlight_edge": "#1F2933",
}

NODE_SIZE_RANGE = (70, 320)
SIMPLE_NODE_SIZE = 520
GRAPHVIZ_ENGINE = "neato"


In [ ]:
from textwrap import fill


def finish_panel(ax, title, subtitle=None):
    """Place a title (and optional wrapped subtitle) above the axes in reserved figure space.

    Used for the warm-up multi-panel figure. Later comparison figures use the
    stricter `make_canonical_panel_figure` below, where the drawable axes box
    has a fixed size.
    """
    fig = ax.figure
    pos = ax.get_position()
    fig_w_in, fig_h_in = fig.get_size_inches()

    panel_w_in = max(fig_w_in * pos.width, 2.0)
    title_chars = max(24, int(panel_w_in * 9))
    sub_chars = max(28, int(panel_w_in * 11))

    wrapped_title = fill(title, width=title_chars) if title else None
    wrapped_sub = fill(subtitle, width=sub_chars) if subtitle else None

    def pts_to_fig_y(pt):
        return (pt / 72.0) / fig_h_in

    title_line_pt = TEXT["panel_title"] * 1.18
    sub_line_pt = TEXT["annotation"] * 1.28
    top_margin_pt = 2
    gap_pt = 3 if (wrapped_title and wrapped_sub) else 0

    title_lines = wrapped_title.count("\n") + 1 if wrapped_title else 0
    sub_lines = wrapped_sub.count("\n") + 1 if wrapped_sub else 0
    header_pts = (
        top_margin_pt
        + title_lines * title_line_pt
        + gap_pt
        + sub_lines * sub_line_pt
    )

    header_h = min(pts_to_fig_y(header_pts), pos.height * 0.32)
    new_h = max(pos.height - header_h, pos.height * 0.55)
    ax.set_position([pos.x0, pos.y0, pos.width, new_h])

    cur_top = pos.y0 + pos.height - pts_to_fig_y(top_margin_pt)
    if wrapped_title:
        fig.text(
            pos.x0, cur_top, wrapped_title,
            ha="left", va="top",
            fontsize=TEXT["panel_title"], color="#17212B",
        )
        cur_top -= pts_to_fig_y(title_lines * title_line_pt + gap_pt)
    if wrapped_sub:
        fig.text(
            pos.x0, cur_top, wrapped_sub,
            ha="left", va="top",
            fontsize=TEXT["annotation"], color=DV_PALETTE["gray"],
        )

    ax.set_axis_off()
    ax.set_aspect("equal")
    return ax


def style_panel_axes(ax):
    ax.set_axis_off()
    ax.set_aspect("equal")
    return ax


def wrap_panel_text(title, subtitle, panel_w_in):
    title_chars = max(24, int(panel_w_in * 9))
    sub_chars = max(28, int(panel_w_in * 11))
    wrapped_title = fill(title, width=title_chars) if title else None
    wrapped_sub = fill(subtitle, width=sub_chars) if subtitle else None
    return wrapped_title, wrapped_sub


def canonical_header_height_in(title, subtitle, panel_w_in):
    wrapped_title, wrapped_sub = wrap_panel_text(title, subtitle, panel_w_in)
    title_lines = wrapped_title.count("\n") + 1 if wrapped_title else 0
    sub_lines = wrapped_sub.count("\n") + 1 if wrapped_sub else 0

    title_line_pt = TEXT["panel_title"] * 1.18
    sub_line_pt = TEXT["annotation"] * 1.28
    gap_pt = 3 if (wrapped_title and wrapped_sub) else 0
    top_pad_pt = 3
    bottom_pad_pt = 4

    header_pt = (
        top_pad_pt
        + title_lines * title_line_pt
        + gap_pt
        + sub_lines * sub_line_pt
        + bottom_pad_pt
    )
    return min(header_pt / 72.0, 1.15)


def make_canonical_panel_figure(title, subtitle=None, *, panel_size=None):
    """Create a figure whose drawable network panel is exactly `panel_size` inches.

    Titles and subtitles live in a separate header band so they do not change
    the size of the actual plotting box used for the network.
    """
    if panel_size is None:
        panel_size = STANDARD_PANEL_SIZE

    panel_w_in, panel_h_in = panel_size
    left_in = 0.20
    right_in = 0.06
    bottom_in = 0.08
    top_in = 0.05
    header_in = canonical_header_height_in(title, subtitle, panel_w_in)

    fig_w = left_in + panel_w_in + right_in
    fig_h = bottom_in + panel_h_in + header_in + top_in

    fig = plt.figure(figsize=(fig_w, fig_h))
    ax = fig.add_axes(
        [
            left_in / fig_w,
            bottom_in / fig_h,
            panel_w_in / fig_w,
            panel_h_in / fig_h,
        ]
    )

    wrapped_title, wrapped_sub = wrap_panel_text(title, subtitle, panel_w_in)
    title_line_pt = TEXT["panel_title"] * 1.18
    gap_pt = 3 if (wrapped_title and wrapped_sub) else 0
    title_lines = wrapped_title.count("\n") + 1 if wrapped_title else 0

    title_top_y = (bottom_in + panel_h_in + header_in - (3 / 72.0)) / fig_h
    fig.text(
        left_in / fig_w,
        title_top_y,
        wrapped_title,
        ha="left",
        va="top",
        fontsize=TEXT["panel_title"],
        color="#17212B",
    )

    if wrapped_sub:
        subtitle_top_y = (
            bottom_in
            + panel_h_in
            + header_in
            - ((3 + title_lines * title_line_pt + gap_pt) / 72.0)
        ) / fig_h
        fig.text(
            left_in / fig_w,
            subtitle_top_y,
            wrapped_sub,
            ha="left",
            va="top",
            fontsize=TEXT["annotation"],
            color=DV_PALETTE["gray"],
        )

    return fig, ax


def normalize_positions(pos, padding=0.08):
    """Rescale a `pos` dict into the unit square with the given padding on both axes.

    A shared coordinate frame is what makes overlap measurements comparable
    across layout algorithms later in the notebook.
    """
    nodes = list(pos)
    coords = np.asarray([pos[node] for node in nodes], dtype=float)
    mins = coords.min(axis=0)
    maxs = coords.max(axis=0)
    center = (mins + maxs) / 2
    span = max(float(np.max(maxs - mins)), 1e-9)
    coords = (coords - center) / span
    coords = 0.5 + coords * (1 - 2 * padding)
    return {node: coords[i] for i, node in enumerate(nodes)}


def compute_axis_limits(positions, padding=0.08):
    """Find shared `(xlim, ylim)` that comfortably fits every layout in an iterable."""
    coords = np.vstack(
        [np.asarray([pos[node] for node in pos], dtype=float) for pos in positions]
    )
    mins = coords.min(axis=0)
    maxs = coords.max(axis=0)
    span = np.maximum(maxs - mins, 1e-9)
    x_pad = span[0] * padding
    y_pad = span[1] * padding
    return (mins[0] - x_pad, maxs[0] + x_pad), (mins[1] - y_pad, maxs[1] + y_pad)


def frame_positions(pos, xlim, ylim):
    """Map coordinates defined inside `(xlim, ylim)` onto the unit square [0, 1]²."""
    nodes = list(pos)
    coords = np.asarray([pos[node] for node in nodes], dtype=float)
    x_span = max(xlim[1] - xlim[0], 1e-9)
    y_span = max(ylim[1] - ylim[0], 1e-9)
    coords[:, 0] = (coords[:, 0] - xlim[0]) / x_span
    coords[:, 1] = (coords[:, 1] - ylim[0]) / y_span
    return {node: coords[i] for i, node in enumerate(nodes)}


def draw_layout_panel(
    ax,
    G,
    pos,
    *,
    title=None,
    subtitle=None,
    node_sizes,
    highlight_nodes=None,
    diagnostic_nodes=None,
    xlim=None,
    ylim=None,
    header_mode="internal",
):
    """Draw one layout panel, optionally overlaying the nodes counted by the benchmark."""
    nx.draw_networkx_edges(
        G, pos=pos,
        edge_color=LAYOUT_STYLE["edge_color"],
        alpha=LAYOUT_STYLE["edge_alpha"],
        width=LAYOUT_STYLE["edge_width"],
        ax=ax,
    )
    nx.draw_networkx_nodes(
        G, pos=pos,
        node_size=[node_sizes[node] for node in G.nodes()],
        node_color=LAYOUT_STYLE["node_fill"],
        edgecolors=LAYOUT_STYLE["node_edge"],
        linewidths=LAYOUT_STYLE["node_linewidth"],
        ax=ax,
    )
    if diagnostic_nodes:
        nx.draw_networkx_nodes(
            G, pos=pos,
            nodelist=diagnostic_nodes,
            node_size=[node_sizes[node] for node in diagnostic_nodes],
            node_color=["#FEE2E2"] * len(diagnostic_nodes),
            edgecolors="#C2410C",
            linewidths=1.6,
            ax=ax,
        )
    if highlight_nodes:
        nx.draw_networkx_nodes(
            G, pos=pos,
            nodelist=highlight_nodes,
            node_size=[node_sizes[node] * 1.05 for node in highlight_nodes],
            node_color=[LAYOUT_STYLE["highlight_fill"]] * len(highlight_nodes),
            edgecolors=LAYOUT_STYLE["highlight_edge"],
            linewidths=1.0,
            ax=ax,
        )
    if xlim is not None:
        ax.set_xlim(*xlim)
    if ylim is not None:
        ax.set_ylim(*ylim)
    style_panel_axes(ax)
    if header_mode == "internal":
        return finish_panel(ax, title, subtitle)
    return ax


## Warm-up: Meet the Main NetworkX Layouts

Before we study overlap on a larger network, it helps to build intuition on a graph we can fully understand.

The next example uses a tiny custom graph with:
- one triangle on the left
- one triangle on the right
- a bridge edge connecting the two groups
- one pendant node hanging from the right triangle

That structure is small enough to read directly, so we can focus on what the layout functions are doing.


In [ ]:
simple_G = nx.Graph()
simple_G.add_edges_from(
    [
        ("A", "B"),
        ("B", "C"),
        ("C", "A"),
        ("C", "D"),
        ("D", "E"),
        ("E", "F"),
        ("F", "D"),
        ("E", "G"),
    ]
)

simple_node_order = list(simple_G.nodes())
simple_node_sizes = {node: SIMPLE_NODE_SIZE for node in simple_node_order}
simple_labels = {node: node for node in simple_node_order}

display(pd.DataFrame(
    {
        "node": simple_node_order,
        "degree": [simple_G.degree(node) for node in simple_node_order],
    }
))


### First Use Guide: `pos` and the Main NetworkX Layout Functions

Every drawing function in NetworkX needs a **position dictionary**, usually called `pos`:

```
{"A": (x, y), "B": (x, y), ...}
```

The layout function is the step that creates those coordinates. It does **not** change the graph itself — only where the same nodes and edges are drawn.

In this warm-up we compare four common choices:

| Function | One-line summary |
|---|---|
| `nx.spring_layout(G, seed=...)` | Force-directed (attract along edges, repel all pairs). Stochastic. |
| `nx.kamada_kawai_layout(G)` | Preserves graph-theoretic distances via optimisation. Deterministic. |
| `nx.circular_layout(G)` | Places nodes evenly on a circle. Baseline. |
| `nx.shell_layout(G, nlist=...)` | Concentric rings for known groupings / hierarchies. |

The next two deep-dive sections unpack **spring** and **Kamada-Kawai** in detail — these are the two most commonly used layouts for real analysis, and they make very different assumptions about what *"close"* should mean in the drawing. The remaining two (circular, shell) are simple enough that the table above is all you need.


### Deep Dive: How `nx.spring_layout` Actually Works

**The physical analogy.** Imagine every node is a charged particle that repels every other charged particle. Now add springs along the graph's edges that pull connected nodes together. Release the whole system and let it settle. That settled configuration is the layout.

This is the **Fruchterman–Reingold** algorithm (1991), the default that `nx.spring_layout` implements.

**The two forces.** At every iteration the algorithm computes, for each node, the net force from two sources:

$$
F_{\text{attract}}(u, v) \;=\; \frac{d(u, v)^2}{k} \qquad \text{(along every edge $(u,v)$)}
$$

$$
F_{\text{repel}}(u, v) \;=\; \frac{k^2}{d(u, v)} \qquad \text{(between every pair of nodes)}
$$

where $d(u, v)$ is the current Euclidean distance between the two nodes in the drawing, and $k$ is the algorithm's **ideal edge length** parameter (more on $k$ below).

- The **attractive** force grows with distance (like a spring: the farther apart, the harder it pulls back).
- The **repulsive** force falls with distance (like electric charge: the closer, the harder it pushes apart).

The equilibrium point of a single edge in isolation is when the two forces balance, which happens exactly at $d(u, v) = k$. That is why $k$ is called the ideal edge length.

**The iteration loop.**

```text
initialise positions randomly
for t in 1..iterations:
    for each node u:
        f_u  = sum of F_repel(u, v) over all other nodes v
        f_u += sum of F_attract(u, v) over every edge (u, v)
        move u by step · (f_u / |f_u|)     # unit vector × step
    step *= cooling_factor                  # temperature drops each iteration
```

The **cooling schedule** (`step *= cooling_factor`) is how the algorithm "simulated-anneals" into a low-energy configuration: early iterations make big moves to escape the random initialisation; later iterations make tiny moves to settle.

**The `k` parameter — what you actually tune.**

- `k` is the *preferred distance* between any pair of nodes at equilibrium.
- Default: $k \approx 1 / \sqrt{N}$ for a graph with $N$ nodes.
- **Smaller `k`** → repulsion falls off faster → layout stays compact (nodes pack closer).
- **Larger `k`** → repulsion reaches further → nodes push each other into more empty space.

⚠️ A subtle footgun: `spring_layout` by default **rescales its final output to fit a unit box**. If you compare `k=0.05` and `k=0.25` with default rescaling, the figures can look suspiciously similar — the end-of-run rescale hides the effect you were trying to show. To see the real geometric difference, pass `scale=None`. We use that trick in Step 3.

**What the algorithm cannot see.**

- It optimises *edge-level* spring energy; it has **no notion of how large matplotlib will render each node marker**. So two large hub markers happily end up overlapping in display space even though their abstract positions are "fine."
- It is **stochastic**: different random seeds produce different layouts. Always pass `seed=` for reproducibility (we use `RANDOM_SEED` from `dataviz_utils`).
- It scales well in practice — roughly $O(N^2)$ per iteration, so typically usable up to a few thousand nodes in a notebook.

**When to reach for it.** Spring is the right default when (a) you have no external structure to impose, (b) the graph is small-to-medium, and (c) you want a drawing that broadly reflects "strongly interconnected things stay close, weakly connected things push apart." It is our workhorse in the rest of this notebook.


### Deep Dive: How `nx.kamada_kawai_layout` Actually Works

**The idea in one sentence.** Place the nodes so that the Euclidean distance between any two nodes in the drawing is as close as possible to their **graph-theoretic distance** (the length of the shortest path through the edges).

This is the **Kamada–Kawai** algorithm (1989). Unlike spring, it is not a simulation — it is an **optimisation** that directly minimises a global energy function.

**The objective.** Let $d_{G}(u, v)$ be the shortest-path distance between nodes $u$ and $v$ on the graph. Let $p_u$ be the drawing position of node $u$. The algorithm minimises

$$
E \;=\; \sum_{u < v} \, \frac{1}{d_{G}(u, v)^2} \, \Big(\, \|p_u - p_v\| - d_{G}(u, v) \,\Big)^2
$$

In words: **"take every pair of nodes; the Euclidean distance between them in the drawing should equal their shortest-path distance in the graph; penalise the squared mismatch, weighted by $1 / d_G^2$ so that *close* pairs matter more than *far* pairs."**

The weighting matters. Without it, distant pairs dominate the sum and the layout would try to get the graph *diameter* right at the cost of every local neighbourhood. The $1 / d_G^2$ weighting says: "local structure first, global scaffolding second."

**The algorithm, at a glance.**

```text
1. compute D = all-pairs shortest-path matrix (BFS from every node)
2. initialise positions (circle by default)
3. while any node has non-zero gradient of E:
       pick the node with the largest gradient
       move it by one Newton step toward reducing E
```

The outer loop is a per-node Newton descent. Because $E$ is smooth and convex-like in practice, it converges fast — typically in $O(N)$ outer iterations for reasonable graphs.

**Why it is deterministic.** No random initialisation step, no temperature, no stochastic component. Run it twice and you get the same layout. This is great for teaching figures and for comparing versions of a network side-by-side.

**The cost.** All-pairs shortest paths is $O(N \cdot (N + E))$ just to build the distance matrix, plus the optimisation on top. Spring is dramatically cheaper on graphs with more than a few thousand nodes.

**What the priors give you.**

- **Bridge edges look long.** Two densely connected blobs separated by a single bridge edge will sit visibly apart, because the graph-distance between any node in one blob and any node in the other is *at least* the bridge's length.
- **Central/peripheral structure survives.** Nodes that are close in graph-theoretic distance land close in the drawing, even when they aren't directly connected.

**What the priors cost you.**

- **Disconnected graphs are undefined.** `d_G(u, v) = ∞` for nodes in different components, which breaks the objective. `nx.kamada_kawai_layout` handles this by laying out each component separately, but the relative placement between components is arbitrary.
- **Like spring, it ignores node size.** The optimiser knows about graph-theoretic distances, not matplotlib marker areas. Overlap in rendered space can still happen.
- **Graph-distance is uniform unit-edge by default.** If your edges carry weights (e.g. stronger interactions), you'd want to pass them. `nx.kamada_kawai_layout` accepts `weight="weight"` to honour edge weights when computing $d_G$.

**Rule of thumb.**
- Use **Kamada-Kawai** when you want a reproducible layout that respects topological distance, your graph is small/medium, and you are making a teaching figure or a side-by-side comparison.
- Use **spring** when you want a quick default on a large-ish graph, or when graph-theoretic distance is not actually the story you are telling.


In [ ]:
simple_shell_groups = [["A", "B", "C"], ["D", "E", "F"], ["G"]]

simple_layout_positions = {
    "Spring layout": normalize_positions(
        nx.spring_layout(simple_G, seed=RANDOM_SEED)
    ),
    "Kamada-Kawai layout": normalize_positions(
        nx.kamada_kawai_layout(simple_G)
    ),
    "Circular layout": normalize_positions(nx.circular_layout(simple_G)),
    "Shell layout": normalize_positions(
        nx.shell_layout(simple_G, nlist=simple_shell_groups)
    ),
}


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11.6, 8.8))
# Reserve spacing *before* drawing so finish_panel sees the final axes positions.
fig.subplots_adjust(top=0.90, bottom=0.03, left=0.04, right=0.98, hspace=0.32, wspace=0.15)

layout_subtitles = {
    "Spring layout": "Force-directed. Good general-purpose default.",
    "Kamada-Kawai layout": "Preserves graph-theoretic distances.",
    "Circular layout": "Uniform ring. Weak for analysis, useful as a baseline.",
    "Shell layout": "Concentric rings around known groups or levels.",
}

# Each layout normalises its coordinates into roughly [0.08, 0.92]² via
# normalize_positions, so a shared axis window fits every panel.
for ax, (layout_name, pos) in zip(axes.flat, simple_layout_positions.items()):
    nx.draw_networkx_edges(
        simple_G, pos,
        edge_color=LAYOUT_STYLE["edge_color"],
        width=LAYOUT_STYLE["edge_width"],
        alpha=LAYOUT_STYLE["edge_alpha"],
        ax=ax,
    )
    nx.draw_networkx_nodes(
        simple_G, pos,
        node_color=[LAYOUT_STYLE["node_fill"] for _ in simple_node_order],
        edgecolors=LAYOUT_STYLE["node_edge"],
        linewidths=LAYOUT_STYLE["node_linewidth"],
        node_size=[simple_node_sizes[node] for node in simple_node_order],
        ax=ax,
    )
    nx.draw_networkx_labels(
        simple_G, pos, labels=simple_labels,
        font_size=11, font_weight="semibold", font_color=LAYOUT_STYLE["node_edge"],
        ax=ax,
    )
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    finish_panel(ax, layout_name, layout_subtitles[layout_name])

fig.suptitle(
    "The same small graph, four different layout algorithms",
    x=0.04, y=0.975, ha="left",
    fontsize=TEXT["figure_title"], fontweight="semibold",
)
plt.show()


### Reading the Warm-up

This tiny graph gives us a shared vocabulary for the rest of the notebook. Each layout expresses a different prior about what the drawing should communicate:

- **Spring layout** — the physical-forces model we just unpacked. With no information beyond edges, it is the fair general-purpose default. The two triangles separate because internal edges pull them into compact blobs, and the bridge edge stretches between them.
- **Kamada-Kawai layout** — the optimisation that matches Euclidean distance to graph distance. That makes structural distance legible: a bridge edge looks long because the two triangles are topologically far from each other.
- **Circular layout** ignores structure entirely and places nodes evenly on a circle. Almost never the best analytical choice, but a useful *baseline*: if a fancier layout does not look meaningfully better than a circle, the fancier layout is not earning its keep.
- **Shell layout** requires external knowledge — a grouping, a levels list, a hierarchy. Right when that structure exists, wrong when it does not.

The deeper lesson: a layout is a *claim* about which relationships the reader should see first. Different algorithms make different claims, and none is universally "correct". From here on we move to a realistic network and ask a more practical follow-up question — even after we pick a layout family, how do we keep the figure readable when encoded node size makes the result crowded?


## Step 1: Load One Graph and Make the Overlap Problem Visible

This notebook uses the built-in Les Misérables co-appearance graph from NetworkX. It is small enough to inspect comfortably, but dense enough that overlap quickly becomes a real visual problem once node size encodes importance.

The first preparation cell does three jobs:
- load the graph once
- build a node-size encoding from degree so overlap has an analytical reason, not a purely artificial one
- expose one easy control, `NODE_SIZE_RANGE`, so students can immediately test how marker size changes both the drawing and the overlap counts

That matters pedagogically. In real figures, overlap usually appears because we try to encode something useful, not because we picked a bad random example.


In [ ]:
G = nx.generators.social.les_miserables_graph()


def map_node_sizes(metric_series, size_range=NODE_SIZE_RANGE):
    low, high = size_range
    value_min = float(metric_series.min())
    value_max = float(metric_series.max())

    if value_min == value_max:
        midpoint = 0.5 * (low + high)
        return {node: float(midpoint) for node in metric_series.index}

    return {
        node: float(np.interp(value, [value_min, value_max], [low, high]))
        for node, value in metric_series.items()
    }


degree_series = pd.Series(dict(G.degree())).sort_values(ascending=False)
node_sizes = map_node_sizes(degree_series, size_range=NODE_SIZE_RANGE)

print(f"Les Misérables graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"node_size range (points²): min={NODE_SIZE_RANGE[0]}, max={NODE_SIZE_RANGE[1]}")
display(pd.DataFrame({"degree": degree_series.head(10)}))


## First Use Guide: Measuring Overlap Before Fixing It

Before we try to improve a layout, it helps to measure the problem explicitly. The workflow is always the same: **quantify first, intervene second**.

### Why we measure in rendered space, not abstract coordinates

Overlap is a **rendering problem**, not a graph-theoretic one:
- NetworkX passes `node_size` to matplotlib as a marker area in points² (1 pt = 1/72 inch).
- Matplotlib figure size is specified in inches.
- The same `pos` dictionary can look perfectly readable on a 6 × 4 in panel and completely crowded on a 4 × 3 in panel — because the same data-space distances become *fewer rendered points* at the smaller size.

The practical consequence: an overlap count is only meaningful if you also state the panel size it was measured at. In this notebook, every reported overlap statistic is measured on **the same panel size used to render the figure**, so the numbers and the images tell the same story. (The exact size is stored once as `STANDARD_PANEL_SIZE` and reused everywhere — it's an implementation detail, not something to memorise.)

### The metric we use throughout the notebook

This notebook reports overlap as **"how many of the N nodes overlap at least one neighbour"**, written as `n_overlap / n_total` (for example `48/77`) or as a percentage. Intuitive reading: *"60% of nodes are still involved in at least one real marker overlap."*

The measurement uses **`min_gap_pt = 0.0`**, so it reports **literal overlap**, not a stricter readability cushion. If you later want a print-oriented clearance benchmark, you can raise `min_gap_pt` above zero — but that would be a different question from "do the markers overlap?"

One more implementation detail: the diagnostic ignores only **tiny floating-point noise** (sub-nanopoint differences). If two circles land exactly tangent after refinement, the measurement treats that as *no overlap*, which is what students should read from the figure.

### What the next helper cell provides

Everything we need is already in `dataviz_utils` (imported via `from dataviz_utils import *` at the top):

- `compute_node_radii(node_sizes)` — convert matplotlib marker areas (points²) into rendered radii in points, accounting for stroke width.
- `overlap_metrics(pos, radii, panel_size=..., ...)` — return a dict with `overlap_node_fraction` (the metric we use) and `overlap_pairs` (kept for completeness).
- `overlap_diagnostics(...)` — like `overlap_metrics` but also returns the *list* of nodes involved in overlap, which powers the coral diagnostic overlay.
- `resolve_node_collisions(...)` — post-process an existing layout using a **size-aware local overlap-removal pass** (used in Step 4).

The next cell adds three notebook-local helpers on top of those library functions — pure display/formatting glue: `overlap_summary` produces the short `"n/N"` strings you see in the DataFrames; `make_overlap_subtitle` and `show_overlap_figure` wire the measurement to the figure title.

The separation between **measurement** (`overlap_metrics`) and **correction** (`resolve_node_collisions`) is important: overlap removal is not a new layout algorithm — it is a post-processing step applied to an existing layout under an explicit display constraint.


In [ ]:
# `compute_node_radii`, `overlap_diagnostics`, `overlap_metrics`,
# `resolve_node_collisions`, `points_per_data_unit`, `STANDARD_PANEL_SIZE`,
# `DISPLAY_GAP_PT`, and `OVERLAP_TOL_PT` all come from `dataviz_utils` via the
# `import *` in the setup cell. This cell only defines the three notebook-local
# helpers that wire the library measurements into this notebook's figures.


def overlap_summary(
    pos,
    radii,
    *,
    panel_size=STANDARD_PANEL_SIZE,
    xlim=(0.0, 1.0),
    ylim=(0.0, 1.0),
):
    """Format `overlap_diagnostics` as the compact 'N/total' string used in tables."""
    diagnostics = overlap_diagnostics(
        pos, radii, panel_size=panel_size, xlim=xlim, ylim=ylim,
    )
    n_total = len(pos)
    n_touch = len(diagnostics["overlap_nodes"])
    return {
        "nodes overlapping another node": f"{n_touch}/{n_total}",
        "fraction": round(diagnostics["overlap_node_fraction"], 2),
    }


def make_overlap_subtitle(
    pos,
    radii,
    *,
    note,
    panel_size=STANDARD_PANEL_SIZE,
    xlim=(0.0, 1.0),
    ylim=(0.0, 1.0),
    diagnostic=False,
):
    """Compose the two-line subtitle used under every overlap figure.

    When there is zero overlap we drop the coral-node explanation — it only
    adds useful context when there are actually coral nodes to look at.
    """
    diagnostics = overlap_diagnostics(
        pos, radii, panel_size=panel_size, xlim=xlim, ylim=ylim,
    )
    n_total = len(pos)
    n_touch = len(diagnostics["overlap_nodes"])
    benchmark_note = (
        "Coral nodes are the nodes the canonical benchmark still counts as truly overlapping. "
        if diagnostic and n_touch > 0 else ""
    )
    subtitle = (
        f"{n_touch}/{n_total} nodes overlap another node "
        f"({diagnostics['overlap_node_fraction']:.0%}). "
        f"{benchmark_note}{note}"
    )
    return diagnostics, subtitle


def show_overlap_figure(
    G,
    pos,
    *,
    title,
    note,
    node_sizes,
    radii,
    panel_size=STANDARD_PANEL_SIZE,
    diagnostic=False,
    xlim=(0.0, 1.0),
    ylim=(0.0, 1.0),
):
    """Measure and render a network panel at the same size, with an optional overlap overlay."""
    diagnostics, subtitle = make_overlap_subtitle(
        pos, radii, note=note, panel_size=panel_size, xlim=xlim, ylim=ylim,
        diagnostic=diagnostic,
    )
    fig, ax = make_canonical_panel_figure(title, subtitle, panel_size=panel_size)
    draw_layout_panel(
        ax,
        G,
        pos,
        node_sizes=node_sizes,
        diagnostic_nodes=diagnostics["overlap_nodes"] if diagnostic else None,
        xlim=xlim,
        ylim=ylim,
        header_mode="external",
    )
    plt.show()


In [ ]:
# Hand-placed positions inside [0, 1]² — two obvious pairs + two lonely nodes.
toy_pos = {
    "A": np.array([0.22, 0.72]),
    "B": np.array([0.28, 0.72]),   # very close to A
    "C": np.array([0.62, 0.72]),   # alone
    "D": np.array([0.38, 0.28]),   # alone
    "E": np.array([0.78, 0.28]),
    "F": np.array([0.84, 0.30]),   # very close to E
}

TOY_MARKER_AREA = 900              # matplotlib node_size in points² (uniform for clarity)
toy_sizes = {node: TOY_MARKER_AREA for node in toy_pos}
toy_radii = compute_node_radii(toy_sizes)

toy_panel_size = STANDARD_PANEL_SIZE  # measure at the exact size we will render at
toy_metrics = overlap_metrics(toy_pos, toy_radii, panel_size=toy_panel_size)

n_nodes = len(toy_pos)
n_overlapping = int(round(toy_metrics["overlap_node_fraction"] * n_nodes))
print(f"{n_overlapping}/{n_nodes} nodes overlap another node ({toy_metrics['overlap_node_fraction']:.0%}).")

fig, ax = plt.subplots(figsize=toy_panel_size)
for name, (x, y) in toy_pos.items():
    ax.scatter(x, y, s=TOY_MARKER_AREA,
               facecolor=LAYOUT_STYLE["node_fill"], edgecolor=LAYOUT_STYLE["node_edge"],
               linewidths=LAYOUT_STYLE["node_linewidth"], zorder=2)
    ax.text(x, y, name, ha="center", va="center",
            fontsize=11, fontweight="semibold", color=LAYOUT_STYLE["node_edge"], zorder=3)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect("equal")
ax.set_axis_off()
ax.set_title(
    f"Toy example — {n_overlapping}/{n_nodes} nodes overlap another node "
    f"({toy_metrics['overlap_node_fraction']:.0%})",
    loc="left", fontsize=TEXT["panel_title"], color="#17212B",
)


### Sanity check: a toy example you can count by eye

Before we run `overlap_metrics` on a 77-node graph where you can't audit the result, let's verify it on a tiny synthetic scene you can check manually.

Six disks placed by hand inside a unit square at a specific panel size:
- `A` and `B` are placed very close together — **they overlap**.
- `E` and `F` are placed very close together — **they overlap**.
- `C` and `D` sit alone, far from everything.

By eye we expect **4 of 6 nodes** (A, B, E, F) to overlap another node — that is the metric the notebook uses throughout. The cell above draws the scene at *exactly* the panel size used for measurement, so what you see on screen is what the metric sees.

The lesson from this toy:

> **An overlap count is only meaningful at the panel size it was measured at.** In this notebook, measurement and rendering always use the same panel, so the numbers and the figures stay in agreement.


## Step 2: Start With Weak Default Layouts

The first comparison is intentionally weak. It uses the default outputs of:
- `nx.spring_layout(...)`
- `nx.kamada_kawai_layout(...)`

Both methods can reveal real structure, but neither one promises non-overlapping nodes. That is the core lesson of this notebook: a reasonable layout and a readable layout are related, but they are not the same thing.

At this stage the figures stay visually simple on purpose:
- every figure is drawn at the same size, so counts and images can be compared directly
- they do **not** use the coral diagnostic yet

The goal is to first see the raw crowding problem before we add the diagnostic overlay that will later mark the nodes still involved in **true overlap**.


In [ ]:
node_radii = compute_node_radii(node_sizes, linewidth=LAYOUT_STYLE["node_linewidth"])

default_spring_pos = normalize_positions(nx.spring_layout(G, seed=RANDOM_SEED))
default_kk_pos = normalize_positions(nx.kamada_kawai_layout(G))

default_overlap_summary = pd.DataFrame(
    [
        {"layout": "Default spring", **overlap_summary(default_spring_pos, node_radii)},
        {"layout": "Default Kamada-Kawai", **overlap_summary(default_kk_pos, node_radii)},
    ]
)
display(default_overlap_summary)


In [ ]:
default_layouts = [
    (
        "Weak choice: default spring layout",
        default_spring_pos,
        "Encoded node size is what makes the crowding visible.",
    ),
    (
        "Weak choice: default Kamada-Kawai layout",
        default_kk_pos,
        "Balanced geometry, still crowded once node size enters.",
    ),
]

for title, pos, note in default_layouts:
    show_overlap_figure(
        G, pos,
        title=title, note=note,
        node_sizes=node_sizes, radii=node_radii,
        diagnostic=False,
    )


## Step 3: Tune the Standard Layout Before Writing Custom Code

The first professional response to overlap should be simple: see whether the built-in layout can be tuned enough to solve most of the problem. Custom collision avoidance, switching engines, or manually nudging nodes are all heavier interventions — they change more of the pipeline and are harder to justify if a one-parameter tweak would already be acceptable.

From this point onward, every overlap-learning figure follows the same contract:
- the network is **measured** at the same panel size it is **rendered** at
- **coral nodes** are the nodes the measurement still counts as truly overlapping

### `k` in `spring_layout` — a refresher

From the spring deep-dive above: `k` is the *preferred distance* between nodes in the force simulation. Smaller `k` packs nodes closer; larger `k` pushes them into more empty space. A rough default is $1 / \sqrt{N}$.

One subtle point worth repeating. `spring_layout` normally rescales its output at the very end of the simulation — by default, coordinates are re-fit into a unit box regardless of `k`. That means if you just plot `spring_layout(G, k=small)` and `spring_layout(G, k=large)` side by side, the two figures can look suspiciously similar: the rescaling step has hidden the effect you were trying to show. The comparison below therefore passes `scale=None`, which disables the final rescale so a larger `k` genuinely produces a layout that occupies more area.


In [ ]:
k_specs = [
    ("Default", None),
    ("More space", 0.12),
    ("Too spread", 0.25),
]

k_raw_positions = {}
for label, k_value in k_specs:
    k_raw_positions[label] = nx.spring_layout(
        G,
        k=k_value,
        iterations=50,
        seed=RANDOM_SEED,
        scale=None,
        center=(0, 0),
        threshold=1e-4,
    )

k_xlim, k_ylim = compute_axis_limits(k_raw_positions.values(), padding=0.12)

k_positions = {
    label: frame_positions(pos, k_xlim, k_ylim)
    for label, pos in k_raw_positions.items()
}

k_summary = pd.DataFrame(
    [
        {
            "setting": label,
            "k": "default" if k_value is None else k_value,
            **overlap_summary(k_positions[label], node_radii),
        }
        for label, k_value in k_specs
    ]
)
display(k_summary)


In [ ]:
title_map = {
    "Default": "1. Default `k`",
    "More space": "2. Larger `k` (0.12)",
    "Too spread": "3. Much larger `k` (0.25)",
}
note_map = {
    "Default": "Compact but crowded.",
    "More space": "Breathing room returns.",
    "Too spread": "Topology starts to feel stretched.",
}

for label, _ in k_specs:
    show_overlap_figure(
        G, k_positions[label],
        title=title_map[label], note=note_map[label],
        node_sizes=node_sizes, radii=node_radii,
        diagnostic=True,
        xlim=(0.0, 1.0), ylim=(0.0, 1.0),
    )


### Reading the `k` Comparison

Because each figure is measured and rendered at the same size, the progression is easy to trust:

- **Default `k`** is a fair baseline, but the coral nodes show that much of the center still contains true node overlaps.
- **Larger `k`** gives the nodes more room while keeping the overall structure recognizable.
- **Much larger `k`** pushes true overlap down further, but the graph starts to feel too stretched for a compact print figure.

Design lesson: reducing overlap is useful, but more empty space is not automatically better. A good layout balances separation with structural coherence.


### Step 3b: Let the Force Simulation Converge

Changing `k` changes the target spacing of the layout. The other natural question is whether the same layout simply needs more time to settle.

Recall from the spring deep-dive: each iteration recomputes attractive and repulsive forces and applies a cooled step. Too few iterations and the layout is still close to the random initialisation; too many iterations and you are paying compute for moves so small they don't change anything visible.

The next sequence keeps the same layout family and the same node sizes, but varies the number of iterations. The diagnostic overlay stays on, so the comparison is still read against the same test: coral nodes are the nodes that remain in true overlap at the rendered panel size.


In [ ]:
iteration_specs = [1, 20, 200]

iteration_raw_positions = {}
for iterations in iteration_specs:
    iteration_raw_positions[iterations] = nx.spring_layout(
        G,
        k=0.08,
        iterations=iterations,
        seed=RANDOM_SEED,
    )

iteration_xlim, iteration_ylim = compute_axis_limits(
    iteration_raw_positions.values(),
    padding=0.10,
)

iteration_positions = {
    iterations: frame_positions(pos, iteration_xlim, iteration_ylim)
    for iterations, pos in iteration_raw_positions.items()
}

iteration_summary = pd.DataFrame(
    [
        {"iterations": iterations, **overlap_summary(iteration_positions[iterations], node_radii)}
        for iterations in iteration_specs
    ]
)
display(iteration_summary)


In [ ]:
iteration_notes = {
    1: "Barely past the random initialization.",
    20: "The large groups are stable, but local crowding remains.",
    200: "Well past convergence: the geometry is stable, but overlap is not gone.",
}

for iterations in iteration_specs:
    show_overlap_figure(
        G, iteration_positions[iterations],
        title=f"Spring layout: {iterations} iterations",
        note=iteration_notes[iterations],
        node_sizes=node_sizes, radii=node_radii,
        diagnostic=True,
        xlim=(0.0, 1.0), ylim=(0.0, 1.0),
    )


In [ ]:
# Lock in one tuned spring layout for reuse in the steps that follow.
tuned_spring_pos = normalize_positions(
    nx.spring_layout(G, k=0.08, iterations=50, seed=RANDOM_SEED)
)


## Step 4: Add a Pure-Python Size-Aware Weighted Relaxation Pass

Tuning helps, but it does not guarantee separation. Force-directed layouts optimize for a global balance of attraction and repulsion — they have **no notion of how large matplotlib will render each node**, so they happily produce layouts where two large hub markers overlap in display space even though their abstract positions are "fine".

If overlap still blocks readability after tuning, the next step is a small post-processing pass in Python: a *display-aware correction* applied on top of an existing layout.

### The Big Idea

Imagine the tuned layout as a bag of differently-sized marbles that were dropped onto a table and froze in place. Some pairs are touching or overlapping. Our goal is not to rearrange the pattern — it is to **gently push overlapping pairs apart, just enough, while preserving the structure the layout already found**.

Three principles capture the whole idea:

1. **It is a massage, not a re-layout.** Nodes stay close to where the original layout placed them. We only nudge pairs that are actually overlapping, by exactly the amount needed to separate them. Everything the layout algorithm already got right — community structure, bridge edges, the overall silhouette — is preserved.

2. **Big nodes don't move much, small nodes do.** If a large hub marker and a small leaf marker overlap, the leaf should slide out of the way and the hub should barely flinch. Students often expect equal 50/50 splitting; the whole point of *size-aware* relaxation is that equal splitting is wrong. A hub with ten neighbours is a visual anchor — moving it disrupts ten edges. Moving a leaf disrupts one.

3. **Everyone reacts to everyone at once.** A naive implementation would push pair A-B apart, commit the move, then push pair A-C apart, commit, and so on. The problem: fixing A-B might *create* a new overlap A-D, and the whole process becomes path-dependent and jittery. Instead, we **compute the total push on each node from all its current overlaps**, apply them all together in one step, then repeat. The motion is smoother, the result is stable, and the order we visit pairs doesn't matter.

### In Concrete Terms

- **Input**: an already-decent layout (the tuned spring from Step 3), the matplotlib marker size of each node, and the panel size the figure will be rendered at.
- **Measurement**: at that panel size, convert marker areas into rendered radii and find every pair of nodes whose centres are closer than the sum of their radii.
- **Correction**: for each such pair, compute the exact displacement needed to separate them, split inversely to node mass (big = little move, small = big move), and accumulate these displacements across all overlapping pairs.
- **Apply**: update every node position by its accumulated displacement. Repeat until no pair overlaps any more.
- **Output**: a new layout where no two markers overlap in rendered space, and every node is as close as possible to where the original layout put it.

### How `resolve_node_collisions` works

The algorithm is a size-aware weighted relaxation loop:

```
anchors ← original tuned layout
coords  ← working copy of anchors
repeat:
    displacement ← zero vector for each node

    for each overlapping pair (i, j):
        direction ← unit vector from i to j
        overlap   ← required_distance − actual_distance

        # move small nodes more, large nodes less
        move_i ← overlap · mass_j / (mass_i + mass_j)
        move_j ← overlap · mass_i / (mass_i + mass_j)

        displacement[i] -= move_i · direction
        displacement[j] += move_j · direction

    coords ← coords + displacement
    stop when no overlap-driven movement remains

# After convergence: translate the whole layout so it sits centered
# inside the rendering frame. Translation preserves every pairwise
# distance, so it cannot reintroduce overlap.
```

The mass split `mass_j / (mass_i + mass_j)` is just the physicist's **lever rule**: in a two-body collision, each body absorbs a share of the displacement proportional to the *other* body's mass. Here "mass" is a power of node radius (default `radius²`), so a node with twice the radius has four times the mass and barely moves.

### Connection to the literature

This is a teaching algorithm, not a full PRISM, VPSC, or GTree implementation. The logic is aligned with the broader literature:
- node overlap should be treated as a rendered-space problem
- node size should affect displacement
- overlap removal is a post-processing step, not a replacement for layout choice

References:
- Gansner & Hu, *Efficient, Proximity-Preserving Node Overlap Removal* — [paper](https://jgaa.info/index.php/jgaa/article/view/paper198)
- Graphviz overlap docs — [overlap](https://graphviz.org/docs/attrs/overlap/)
- Nachmanson et al., *Node Overlap Removal by Growing a Tree* — [paper](https://jgaa.info/index.php/jgaa/article/view/paper442)


In [ ]:
size_aware_pos = resolve_node_collisions(
    tuned_spring_pos, node_radii,
    panel_size=STANDARD_PANEL_SIZE,
)

collision_summary = pd.DataFrame(
    [
        {"layout": "Tuned spring",
         **overlap_summary(tuned_spring_pos, node_radii)},
        {"layout": "Size-aware weighted refinement",
         **overlap_summary(size_aware_pos, node_radii)},
    ]
)
display(collision_summary)


In [ ]:
show_overlap_figure(
    G, tuned_spring_pos,
    title="Tuned spring layout",
    note="A better baseline, but several coral nodes still overlap at the rendered panel size.",
    node_sizes=node_sizes, radii=node_radii,
    diagnostic=True,
)

show_overlap_figure(
    G, size_aware_pos,
    title="Stronger choice: size-aware weighted refinement",
    note="After refinement, the measurement reports zero true overlaps at the rendered panel size.",
    node_sizes=node_sizes, radii=node_radii,
    diagnostic=True,
)


## Step 5: Compare With a Graphviz Alternative

Pure Python keeps the whole workflow inside the notebook, but it is not the only strong option. Graphviz is a mature, external layout engine with decades of tuning behind it, and it is worth knowing as a point of comparison.

### First Use Guide: `graphviz_layout_plain(...)`

The helper below:
- builds a `pygraphviz.AGraph` object directly in Python
- assigns Graphviz graph, node, and edge attributes in that object
- asks a chosen Graphviz engine (`neato` here) to compute the layout
- asks Graphviz to run **`overlap="vpsc"`**, a quadratic optimization method designed to remove node overlaps while keeping displacement small
- reads back node coordinates from the `pos` attributes that Graphviz writes onto the nodes

A subtle but important lesson appears here: **Graphviz removes overlaps in its own coordinate system.** If we take those coordinates and then squeeze them back into the notebook's unit square, we compress the drawing and can reintroduce overlap that Graphviz had already removed.

For that reason, the Graphviz figure is shown and measured in **Graphviz's own coordinate system**, not forced into the same frame as the NetworkX layouts. This keeps the teaching message clear:
- the notebook's Python method is a **teaching-scale local, size-aware relaxation**
- Graphviz's `vpsc` is a **mature external overlap-removal engine** that should be judged on the geometry it actually produces

⚠️ **Pygraphviz is not required to run the rest of the notebook.** The import is deferred so that readers without pygraphviz installed can still go through Steps 1–4. If `import pygraphviz` fails, this cell will print a graceful message and skip Step 5.

References:
- Graphviz overlap docs: [overlap](https://graphviz.org/docs/attrs/overlap/)
- Graphviz VPSC option: documented under the same `overlap` attribute reference


In [ ]:
def graphviz_layout_plain(G, node_sizes, *, engine=GRAPHVIZ_ENGINE):
    """Compute a Graphviz layout through PyGraphviz and keep Graphviz's native coordinates.

    `pygraphviz` is imported here, not at the top of the notebook, so that the
    rest of the notebook runs on installs without it.
    """
    import pygraphviz as pgv

    max_size = max(node_sizes.values())

    A = pgv.AGraph(strict=False, directed=False)
    A.graph_attr.update(overlap="vpsc", outputorder="edgesfirst", splines="false", sep="+6")
    A.node_attr.update(
        shape="circle",
        fixedsize="true",
        style="filled",
        fillcolor=LAYOUT_STYLE["node_fill"],
        color=LAYOUT_STYLE["node_edge"],
        fontsize="1",
        penwidth=str(LAYOUT_STYLE["node_linewidth"]),
    )
    A.edge_attr.update(
        color=LAYOUT_STYLE["edge_color"],
        penwidth=str(LAYOUT_STYLE["edge_width"]),
    )

    for node in G.nodes():
        size_ratio = math.sqrt(node_sizes[node] / max_size)
        width = 0.18 + 0.20 * size_ratio
        A.add_node(node, width=f"{width:.4f}", height=f"{width:.4f}", label=" ")

    for left, right in G.edges():
        A.add_edge(left, right)

    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message=r"Warning: node \'.*\'[, ].*size too small for label.*",
            category=RuntimeWarning,
            module=r"pygraphviz\.agraph",
        )
        A.layout(prog=engine)

    positions = {}
    for node in G.nodes():
        pos_attr = A.get_node(node).attr.get("pos")
        if not pos_attr:
            raise RuntimeError(f"Graphviz did not assign a position to node {node!r}.")
        x_str, y_str, *_ = pos_attr.split(",")
        positions[node] = np.array([float(x_str), float(y_str)], dtype=float)

    if set(positions) != set(G.nodes()):
        raise RuntimeError("Graphviz did not return a complete set of node positions.")

    return positions


In [ ]:
try:
    graphviz_pos = graphviz_layout_plain(G, node_sizes, engine=GRAPHVIZ_ENGINE)
except ImportError:
    print("pygraphviz not installed — skipping the Graphviz row of the scorecard.")
    graphviz_pos = None

if graphviz_pos is not None:
    graphviz_xlim, graphviz_ylim = compute_axis_limits([graphviz_pos], padding=0.06)
    graphviz_panel_size = (5.2, 4.8)

# Each row is scored on the same panel the layout is actually rendered in,
# so "nodes overlapping another node" matches what the eye sees in its figure.
scorecard_layouts = [
    ("Default spring",                  default_spring_pos, (0.0, 1.0), (0.0, 1.0), STANDARD_PANEL_SIZE, "Measured at the rendered panel size"),
    ("Default Kamada-Kawai",            default_kk_pos,     (0.0, 1.0), (0.0, 1.0), STANDARD_PANEL_SIZE, "Measured at the rendered panel size"),
    ("Tuned spring",                    tuned_spring_pos,   (0.0, 1.0), (0.0, 1.0), STANDARD_PANEL_SIZE, "Measured at the rendered panel size"),
    ("Size-aware weighted refinement",  size_aware_pos,     (0.0, 1.0), (0.0, 1.0), STANDARD_PANEL_SIZE, "Measured at the rendered panel size"),
]
if graphviz_pos is not None:
    scorecard_layouts.append(
        ("Graphviz (neato + vpsc)",     graphviz_pos,       graphviz_xlim, graphviz_ylim, graphviz_panel_size, "Measured in Graphviz native coordinates"),
    )

layout_scorecard = pd.DataFrame(
    [
        {
            "benchmark": benchmark,
            "layout": name,
            **overlap_summary(pos, node_radii, panel_size=panel_size, xlim=xlim, ylim=ylim),
        }
        for name, pos, xlim, ylim, panel_size, benchmark in scorecard_layouts
    ]
)
display(layout_scorecard)


In [ ]:
show_overlap_figure(
    G, size_aware_pos,
    title="Pure Python: size-aware weighted refinement",
    note="Measured and rendered at the same panel size; coral would indicate true overlap.",
    node_sizes=node_sizes, radii=node_radii,
    diagnostic=True,
)

if graphviz_pos is not None:
    show_overlap_figure(
        G, graphviz_pos,
        title="Graphviz: neato + vpsc",
        note="Shown in Graphviz's native coordinates.",
        node_sizes=node_sizes, radii=node_radii,
        diagnostic=True,
        panel_size=graphviz_panel_size,
        xlim=graphviz_xlim, ylim=graphviz_ylim,
    )


### Reading the Final Comparison

The final comparison separates **two evaluation contracts**:

- The first four rows of the scorecard — the pure Python / NetworkX layouts — are measured in the unit square where they were produced.
- The Graphviz row (when pygraphviz is available) is measured in **Graphviz's native coordinate system**, because that is the geometry in which its overlap-removal guarantee is defined.

If we normalize the Graphviz coordinates back into the unit square, we compress the layout and can undo part of the spacing Graphviz just created — so we don't.

The final lesson:
- The **size-aware weighted refinement** is the strongest pure-Python method in this notebook and removes true overlap completely.
- **Graphviz (`neato` + `vpsc`)** is a strong external alternative, and on its own geometry it also removes true overlap completely.
- The two **default** layouts are honest starting points, but still produce visible crowding.
- The **tuned spring** layout improves the baseline without extra machinery, but not enough to eliminate overlap.

Two evaluation ideas should stay with the student:
- an in-notebook measurement for methods that live inside the notebook's display frame
- a native external-engine measurement for a method whose guarantee depends on the geometry it produces


## Final Takeaway Checklist

After working through the notebook, a student should be able to answer:

1. What two forces does `spring_layout` balance, and what does the parameter `k` control in each?
2. What objective does `kamada_kawai_layout` minimise, and why does that make it deterministic?
3. Why do nodes still overlap in a reasonable force-directed layout?
4. Why is size-aware overlap removal a post-processing step rather than just "more layout tuning"?
5. Why should large and small nodes not be displaced in exactly the same way?
6. Why is a weighted displacement field preferable to pushing both nodes in an overlap by the same amount?
7. How is a Graphviz overlap-removal pass different from the notebook's pure-Python refinement?

The core lesson is simple:

**Start with a meaningful layout, tune it carefully, measure true overlap at the size you plan to render it, and only then apply a size-aware correction or switch to a stronger external engine.**
